# Step 5 — Tensor-Core FlashAttention-1 (v5_wmma) — run-of-record (vast.ai T4)

Single fused pass, but both matmuls run on Turing's **WMMA tensor cores** (**FP16-in / FP32-accum**): `S = Q@K^T` and `O += P@V` become 16x16x16 GEMM tiles instead of v4's warp-shuffle dot products. Softmax is forced through smem (store S -> row-softmax -> reload P as half) because WMMA accumulators are opaque; the O-rescale add is the tensor core's own `C += A*B` (load running O into the accumulator, mma P@V on top).

**Roofline says:** the MMA floor drops ~8x (8192x64: 16.97 ms FP32 -> **2.11 ms** FP16), still MMA-bound. New wrinkle: as the matmul floor collapses, the softmax `exp` (MUFU) grows to ~25% util at d=64.

**Thesis to test:** v5 should *beat v4* in wall-clock by attacking v4's measured FMA-efficiency wall (18x off floor), while keeping S off HBM (~+17 MB peak, like v4). First version NOT bit-comparable to FP32 SDPA -> looser tolerance (atol/rtol 2e-2). Counter-free throughout (ncu blocked on cloud rentals): peak-memory for the S proof, CUPTI for per-kernel timing.


## 0. Bootstrap the vast.ai CUDA devel image (idempotent)
The bare CUDA devel image has **no `python` symlink, no torch, and we start outside the repo** —
the three things that failed on the first run. This cell fixes all of them; safe to re-run.


In [ ]:
# (a) Route EVERYTHING through the Jupyter kernel's own interpreter. `!python` (a shell
#     subprocess) and an in-kernel `import torch` are otherwise DIFFERENT pythons on this
#     image, so torch installed for one is invisible to the other. Point the `python`
#     symlink at sys.executable and install into sys.executable -> one python, both paths.
import subprocess, sys, os
KPY = sys.executable  # the kernel running this notebook
subprocess.run(['ln','-sf', KPY, '/usr/local/bin/python'])
print('python ->', KPY)

# (b) torch (cu124) into the KERNEL interpreter if missing, plus build deps.
try:
    import torch  # noqa: F401
    print('torch already present:', torch.__version__)
except ModuleNotFoundError:
    subprocess.run([KPY,'-m','pip','install','-q','torch',
                    '--index-url','https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([KPY,'-m','pip','install','-q','ninja','pytest'], check=True)

# (c) locate an existing repo checkout (walk up from cwd) or clone, then chdir in.
#     Works whether you uploaded the bare .ipynb or run it from inside a clone.
REPO='https://github.com/gkienpham-cmd/flashattention-cuda.git'
def _find_repo(p):
    while True:
        if os.path.isdir(os.path.join(p,'.git')) and os.path.isdir(os.path.join(p,'fa_kernels')):
            return p
        nxt = os.path.dirname(p)
        if nxt == p: return None
        p = nxt
root = _find_repo(os.getcwd())
if root is None:
    if not os.path.isdir('flashattention-cuda/.git'):
        subprocess.run(['git','clone',REPO], check=True)
    root = os.path.abspath('flashattention-cuda')
os.chdir(root)
subprocess.run(['git','pull','origin','main'], check=True)
print('cwd =', os.getcwd())


## 1. Confirm the toolchain + GPU


In [ ]:
!nvcc --version


In [ ]:
!python -c "import torch; print('cuda_ok', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', torch.version.cuda)"


## 2. Predict the roofline BEFORE running — FP16 floor vs the v4 FP32 floor (the deliverable is the distance)
The FP16 tensor-core peak (65 TFLOPS) is ~8x the FP32 CUDA-core peak (8.1), so the MMA lower bound drops ~8x. Watch the `t_mufu` line: the softmax exp is now a meaningful share at d=64.


In [ ]:
!python -m roofline.predict --arch sm_75 --shape 1x8x8192x64 --precision fp16
!python -m roofline.predict --arch sm_75 --shape 1x8x8192x64 --precision fp32
!python -m roofline.predict --arch sm_75 --shape 1x8x8192x128 --precision fp16


## 3. Build smoke (JIT compile v5 + one forward). A clean compile here = the WMMA kernel built.
Inputs are FP32 (the public `attention()` contract); the v5 host entry casts to half internally. Output is FP32.


In [ ]:
# Clear any stale JIT cache from a prior build, then compile v5 via one forward.
import shutil, os, glob
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v5_wmma')):
    shutil.rmtree(d, ignore_errors=True)
import torch
from fa_kernels import attention
q=torch.randn(1,8,512,64,device='cuda'); k=torch.randn_like(q); v=torch.randn_like(q)
out=attention(q,k,v,backend='v5_wmma'); torch.cuda.synchronize()
print('v5 built + ran, out shape', tuple(out.shape), '| dtype', out.dtype)


## 4. Correctness vs SDPA (FP16 tolerance atol/rtol 2e-2) — full sweep + the long-N O-rescale stability test
v5 is the first version that loosens tolerance: FP16 inputs carry ~2^-11 relative error before the math. A miss here is a *finding* (precision or a real bug), not a knob to widen further.


In [ ]:
!python -m pytest tests/test_correctness.py -k v5_wmma -v


## 5. S-elimination proof (peak memory) — v5 must stay at v4's ~+17 MB, NOT v2's +2164 MB
Tensor cores changed the *math*, not the *schedule*: S is still never materialized. The +17 MB should survive the WMMA rewrite (the half K/V/Q tiles + FP32 oRun live in smem, not HBM).


In [ ]:
%%writefile mem_check_v5.py
import torch
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
for backend in ["v2_tiled", "v4_fused", "v5_wmma"]:
    q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); base=torch.cuda.memory_allocated()
    out=attention(q,k,v,backend=backend); torch.cuda.synchronize()
    print(f"{backend}: peak +{(torch.cuda.max_memory_allocated()-base)/1e6:.1f} MB  (a materialized S = {B*H*N*N*4/1e6:.0f} MB)")
    del q,k,v,out; torch.cuda.empty_cache()


In [ ]:
!python mem_check_v5.py


## 6. CUPTI per-kernel trace — v5 is a SINGLE fused kernel; measure its distance from the 2.11 ms FP16 floor
v4 was one fused kernel ~18x above its 16.97 ms FP32 floor. v5's floor is ~8x lower (2.11 ms). Read v5's total CUDA time: does WMMA close v4's gap, or does a new limiter (MUFU exp / smem-softmax round-trip / occupancy) appear? Note the cast-to-half shows as a separate op.


In [ ]:
%%writefile prof_check_v5.py
import torch
from torch.profiler import profile, ProfilerActivity
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
for _ in range(3): attention(q,k,v,backend="v5_wmma")   # warmup + JIT
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CUDA]) as prof:
    for _ in range(10): attention(q,k,v,backend="v5_wmma")
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))


In [ ]:
!python prof_check_v5.py


## 7. Bench vs SDPA — the headline. Run v5 (fp16), then v4 (fp32) for the apples-to-apples comparison.
Win condition: v5/SDPA speedup > v4/SDPA at matching shapes (v5 beats v4). SDPA runs in the same precision as our inputs, so v5's SDPA baseline is FP16, v4's is FP32 — note that when reading.


In [ ]:
!python -m bench.harness --backend v5_wmma --precision fp16


In [ ]:
!python -m bench.harness --backend v4_fused --precision fp32


## 8. (Optional) causal sweep — exercises the in-tile causal mask in the WMMA path


In [ ]:
!python -m bench.harness --backend v5_wmma --precision fp16 --causal
